In [8]:
import re
import copy

class DefiniteClause:
    """Represents a rule (premises => conclusion) or a simple fact."""
    def __init__(self, premises, conclusion):
        self.premises = premises      # List of tuples: [("American", ("p",)), ...]
        self.conclusion = conclusion  # Tuple: ("Criminal", ("p",))

    def __str__(self):
        premise_str = " ∧ ".join([f"{p}({', '.join(args)})" for p, args in self.premises])
        return f"{premise_str} ⇒ {self.conclusion[0]}({', '.join(self.conclusion[1])})"


def parse_literal(literal_str):
    """Parses a string like 'Sells(Robert, x, A)' into ('Sells', ('Robert', 'x', 'A'))."""
    match = re.match(r"(\w+)\(([^)]+)\)", literal_str.strip())
    if not match:
        raise ValueError(f"Invalid literal format: '{literal_str}'")
    pred = match.group(1)
    # Split by comma and strip whitespaces from arguments
    args = tuple(arg.strip() for arg in match.group(2).split(","))
    return (pred, args)


def parse_rule(rule_str):
    """Parses a string like 'Missile(x) ∧ Owns(A, x) => Sells(Robert, x, A)'."""
    if "=>" not in rule_str:
        raise ValueError("Rules must contain '=>' separating premises and conclusion.")

    premise_part, conclusion_part = rule_str.split("=>")

    # Split premises by ∧ or ^ or AND safely
    premise_strs = re.split(r"∧|\^|AND", premise_part)
    premises = [parse_literal(p) for p in premise_strs if p.strip()]
    conclusion = parse_literal(conclusion_part)

    return DefiniteClause(premises, conclusion)


def is_variable(arg, variables_set):
    """Determines if an argument is a variable based on user-provided variable names."""
    # Also matches standardized variables like 'x_1'
    base_arg = arg.split("_")[0]
    return base_arg in variables_set


def standardize_variables(rule, counter, variables_set):
    """Renames variables in a rule with a unique numeric suffix to avoid name collisions."""
    suffix = f"_{counter}"
    new_premises = []

    for pred, args in rule.premises:
        new_args = tuple(arg + suffix if is_variable(arg, variables_set) else arg for arg in args)
        new_premises.append((pred, new_args))

    c_pred, c_args = rule.conclusion
    new_c_args = tuple(arg + suffix if is_variable(arg, variables_set) else arg for arg in c_args)
    new_conclusion = (c_pred, new_c_args)

    return DefiniteClause(new_premises, new_conclusion)


def match_premises(premises, facts, theta, variables_set):
    """Recursively finds all substitution sets (thetas) that satisfy the premises using current facts."""
    if not premises:
        return [theta]

    first, rest = premises[0], premises[1:]
    pred, args = first
    successful_thetas = []

    for f_pred, f_args in facts:
        if f_pred != pred or len(f_args) != len(args):
            continue

        local_theta = copy.deepcopy(theta)
        possible = True

        for r_arg, f_arg in zip(args, f_args):
            if "_" in r_arg or is_variable(r_arg, variables_set):
                # Variable handling
                if r_arg in local_theta:
                    if local_theta[r_arg] != f_arg:
                        possible = False
                        break
                else:
                    local_theta[r_arg] = f_arg
            else:
                # Constant matching constant
                if r_arg != f_arg:
                    possible = False
                    break

        if possible:
            res = match_premises(rest, facts, local_theta, variables_set)
            successful_thetas.extend(res)

    return successful_thetas


def subst(theta, literal):
    """Applies a substitution dictionary to a literal."""
    pred, args = literal
    new_args = tuple(theta.get(arg, arg) for arg in args)
    return (pred, new_args)


def fol_fc_ask(kb_rules, kb_facts, alpha, variables_set):
    """The dynamic FOL-FC-ASK Forward Chaining Algorithm."""
    facts = set(kb_facts)
    standardization_counter = 0

    print("\n--- Starting Forward Chaining Inference ---")
    iteration = 1

    while True:
        print(f"\n🔄 Iteration {iteration}:")
        new = set()

        for rule in kb_rules:
            standardization_counter += 1
            std_rule = standardize_variables(rule, standardization_counter, variables_set)
            thetas = match_premises(std_rule.premises, facts, {}, variables_set)

            for theta in thetas:
                q_prime = subst(theta, std_rule.conclusion)

                if q_prime not in facts and q_prime not in new:
                    new.add(q_prime)
                    print(f"  Inferred: {q_prime[0]}({', '.join(q_prime[1])})")

                    if q_prime == alpha:
                        print(f"\n🎯 Goal matched successfully: {q_prime[0]}({', '.join(q_prime[1])})")
                        return theta if theta else True

        if not new:
            print("\n❌ No new sentences inferred. Goal unreachable or complete.")
            return False

        facts.update(new)
        iteration += 1


# --- Dynamic User Interface / Input Processing ---
if __name__ == "__main__":
    print("=== DYNAMIC FIRST ORDER LOGIC FORWARD CHAINING ===")

    # 1. Define what symbols are variables
    vars_input = input("Enter variables separated by commas (e.g., p, q, r, x): ")
    variables_set = {v.strip() for v in vars_input.split(",") if v.strip()}

    # 2. Input Rules
    print("\nEnter your implication rules one by one. Enter 'DONE' when finished.")
    print("Format example: American(p) ∧ Weapon(q) ∧ Sells(p,q,r) ∧ Hostile(r) => Criminal(p)")
    rules = []
    while True:
        r_in = input("Rule: ")
        if r_in.strip().upper() == "DONE":
            break
        if r_in.strip():
            try:
                rules.append(parse_rule(r_in))
            except Exception as e:
                print(f"⚠️ Parsing Error: {e}. Please re-enter.")

    # 3. Input Facts
    print("\nEnter your initial ground facts one by one. Enter 'DONE' when finished.")
    print("Format example: Owns(A, T1)")
    initial_facts = []
    while True:
        f_in = input("Fact: ")
        if f_in.strip().upper() == "DONE":
            break
        if f_in.strip():
            try:
                initial_facts.append(parse_literal(f_in))
            except Exception as e:
                print(f"⚠️ Parsing Error: {e}. Please re-enter.")

    # 4. Input Target Query
    print("\nEnter the atomic goal query to prove.")
    print("Format example: Criminal(Robert)")
    while True:
        q_in = input("Query: ")
        if q_in.strip():
            try:
                query = parse_literal(q_in)
                break
            except Exception as e:
                print(f"⚠️ Parsing Error: {e}. Please re-enter.")

    # Run the engine
    result = fol_fc_ask(rules, initial_facts, query, variables_set)
    print(f"\nFinal Algorithm Return Value: {result}")


=== DYNAMIC FIRST ORDER LOGIC FORWARD CHAINING ===
Enter variables separated by commas (e.g., p, q, r, x): p,q,r,x

Enter your implication rules one by one. Enter 'DONE' when finished.
Format example: American(p) ∧ Weapon(q) ∧ Sells(p,q,r) ∧ Hostile(r) => Criminal(p)
Rule: American(p) ∧ Weapon(q) ∧ Sells(p, q, r) ∧ Hostile(r) => Criminal(p)
Rule: Missile(x) ∧ Owns(A, x) => Sells(Robert, x, A)
Rule: Missile(x) => Weapon(x)
Rule: Enemy(x, America) => Hostile(x)
Rule: DONE

Enter your initial ground facts one by one. Enter 'DONE' when finished.
Format example: Owns(A, T1)
Fact: Owns(A, T1)
Fact: Missile(T1)
Fact: American(Robert)
Fact: Enemy(A, America)
Fact: DONE

Enter the atomic goal query to prove.
Format example: Criminal(Robert)
Query: Criminal(Robert)

--- Starting Forward Chaining Inference ---

🔄 Iteration 1:
  Inferred: Sells(Robert, T1, A)
  Inferred: Weapon(T1)
  Inferred: Hostile(A)

🔄 Iteration 2:
  Inferred: Criminal(Robert)

🎯 Goal matched successfully: Criminal(Robert)

F